In [1]:
from scipy.cluster.hierarchy import linkage , dendrogram
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

C:\Users\PGCP-AI.STUDENTSDC\.conda\envs\surprise\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('titles.csv')
df.shape

(5850, 15)

In [3]:
df.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


In [4]:
sentences = [
    "He battted well today in the cricket meatch",
    "Tomorrow will be wednesday",
    "She played excellently yesterday with many goals",
    "Today will be the event",
    "He performed well in that game"
]

In [5]:
model  = SentenceTransformer('BAAI/bge-small-en-v1.5')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1362.13it/s]


In [6]:
embeddings = model.encode(sentences)
embeddings.shape

(5, 384)

In [13]:
similarities = model.similarity(embeddings,embeddings) # similarity between sentences
similarities  
## the model.similarity() function returns tensor array (not a numpy array) 
## so if we want to process it further we have to convert it intp numpy assay by writinh similarities.numpy()

tensor([[1.0000, 0.5402, 0.6310, 0.5531, 0.7714],
        [0.5402, 1.0000, 0.5230, 0.8452, 0.4751],
        [0.6310, 0.5230, 1.0000, 0.5578, 0.7182],
        [0.5531, 0.8452, 0.5578, 1.0000, 0.5212],
        [0.7714, 0.4751, 0.7182, 0.5212, 1.0000]])

## Filtering Indian MOVIES after 1980

In [14]:
df_indian_movies = df[(df['production_countries'].str.contains('IN')) & (df['type']=='MOVIE') & (df['release_year']>=1980)]
df_indian_movies.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
19,tm94651,Dostana,MOVIE,Two close friends decide to enter law enforcem...,1980,NaN,161,"['drama', 'comedy', 'crime', 'romance', 'action']",['IN'],NaN,tt0080653,2.1,25.0,3.980,4.9
56,tm721687,Vaashi,MOVIE,"Ebin Mathew, a budding lawyer ambitiously join...",1983,NaN,123,"['drama', 'thriller']",['IN'],NaN,tt13913068,6.7,388.0,3.790,NaN
65,tm52815,Disco Dancer,MOVIE,"Anil, a street singer, is humiliated and drive...",1982,NaN,106,"['drama', 'romance']",['IN'],NaN,tt0208903,6.4,1476.0,7.879,5.3
66,tm159912,Agneepath,MOVIE,"The movie depicts the life of a young boy, Vij...",1990,NaN,174,"['drama', 'action', 'crime']",['IN'],NaN,tt0098999,7.6,8947.0,4.062,6.5
71,tm157603,Dil,MOVIE,Raja lives a poor lifestyle along with his dad...,1990,NaN,172,"['drama', 'comedy', 'romance']",['IN'],NaN,tt0099429,6.6,5316.0,3.992,6.5


In [15]:
all_desc = list(df_indian_movies['description'])
out_embs = []
for r in tqdm(all_desc):
  embd = model.encode(r)
  out_embs.append(embd)
out_embd = np.array(out_embs)
out_embd.shape

100%|██████████| 570/570 [00:25<00:00, 22.59it/s]


(570, 384)

In [17]:
movies_idx = list(df_indian_movies['title'] + ' - ' + df_indian_movies['release_year'].astype(str))
df_embd = pd.DataFrame(out_embd,index=movies_idx)
df_embd.to_csv('movie-embds.csv')  
# usually we store embeddings in vector database not in csv file ... but just for siplicity, for this code we have saved that into csv file

In [23]:
query = "College guys having fun amd emjoying"
q_emb = model.encode([query])
sims = []
for e in tqdm(out_embs):    ## out_embs is array of embeddings
    sims.append(model.similarity(q_emb,e).numpy()[0])
df_ind = df_indian_movies[['id','title','description','release_year']].copy()
df_ind.loc[:,'sims'] = np.array(sims)
df_ind.sort_values(by= 'sims',ascending = False).head(10)

100%|██████████| 570/570 [00:00<00:00, 6261.99it/s]


,id,title,description,release_year,sims
4565,tm471802,Sur Sapata,This story is about young boys who have capabi...,2019,0.629349
5391,tm1132325,Don,A youngster who is still figuring out what his...,2022,0.621389
972,tm185797,Fukrey,The paths of four dream-chasing college friend...,2013,0.619176
513,tm54147,Dostana,Two straight guys who pretend to be a gay coup...,2008,0.603043
434,tm61056,Dil Chahta Hai,Three inseparable childhood friends are just o...,2001,0.589248
5154,tm1213009,Dear Friend,Four close friends who live together in Bangal...,2022,0.585681
316,tm142564,3 Idiots,Rascal. Joker. Dreamer. Genius... You've never...,2009,0.579595
1455,tm233843,Pyaar Ka Punchnama 2,"After falling in love, three roommates experie...",2015,0.573188
792,tm166740,Chhota Bheem & Krishna in Mayanagari,Bheem and his Friends are having a great time ...,2011,0.572340
442,tm22540,Phir Hera Pheri,"Babu Rao, Raju and Shyam, are living happily a...",2006,0.570162


In [28]:
query = "Damini"
q_emb = model.encode([query])
sims = []
for e in tqdm(out_embs):    ## out_embs is array of embeddings
    sims.append(model.similarity(q_emb,e).numpy()[0])
df_ind = df_indian_movies[['id','title','description','release_year']].copy()
df_ind.loc[:,'sims'] = np.array(sims)
df_ind.sort_values(by= 'sims',ascending = False).head(10)
## Here we have ordered the data by similarity score ... bu in real life recommendation system must use views or ratings to sort the data not 
##   the similarity search only .... because if we just only sort by similarity score then 
##  it may recommend the movie with same description but may not be that good or have less rating or less views so we must sort by views or rating

100%|██████████| 570/570 [00:00<00:00, 6168.11it/s]


,id,title,description,release_year,sims
182,tm184326,Damini,The theme revolves around the character Damini...,1993,0.646144
187,tm47530,Jeans,Vishu and Ramu are a pair of Indian twins livi...,1998,0.556070
1119,tm150962,Mumbai Delhi Mumbai,"Pia, a typical “Mumbai” girl, makes her first ...",2014,0.553191
1065,tm147908,Anaamika,Anamika comes to India in search of her husban...,2014,0.547103
5821,tm1110241,Kaaval,Thampan and Antony are a long time best friend...,2021,0.540650
2942,tm469399,Music Teacher,"Despite his ambitions to make it big, Beni is ...",2018,0.531661
4210,tm840723,Ramprasad Ki Tehrvi,Ramprasad's entire family gathers under one ro...,2019,0.531052
2619,tm368441,Aapla Manus,Sr Inspector Maruti Nagargoje is investigating...,2018,0.514330
792,tm166740,Chhota Bheem & Krishna in Mayanagari,Bheem and his Friends are having a great time ...,2011,0.513346
428,tm72587,Guru,"Gurukant Desai hails from Idhar, a small villa...",2007,0.513013
